# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SabeenSaeed/machine_learning_projects/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window
Lane: Refresh/content-opportunity scoring. One row means one pseudonymized content page for one client on one daily performance snapshot, identified by client_hash_id, content_hash_id, and report_date. I use the mid-panel month March 2026 (2026-03-01 through 2026-03-31). The main source is fact_content_daily_performance, joined to dim_content for content metadata and dim_clients for client history and availability. I rank pages for refresh review using signals that were observable at the decision moment. I deliberately exclude raw or identifying values, product decision flags, and any outcome measured after the decision moment.

In [21]:
# This cell is for CODE (numbers, a query, a check).
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




## 2. Fields: feature / label / context / excluded

Features: March-window search impressions, search clicks, average position, content age, and GA4 sessions/engagement when available. Each is used only as an observed signal available at the decision moment.
Label/proxy: A future-looking refresh-risk proxy: whether the page’s search impressions decline by more than 20% in the next 30-day outcome window after the March feature window. This is a target, never a feature.
Context: client_hash_id, content_hash_id, report_date, url_hash_id, content type, and client history dates. These identify, join, group, or split records; they are not model inputs.
Excluded: trend_direction and trend_pct are excluded because they are derived from the outcome/current comparison window and would leak label information. Raw URLs, titles, client names, and queries are excluded because the release is pseudonymized and they are not needed for this decision.

In [16]:
# This cell is for CODE (numbers, a query, a check).
!pip -q install duckdb pandas scikit-learn

import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "content": f"read_parquet('{REL}/dim_content.parquet')",
    "daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

MONTH_START = "2026-03-01"
MONTH_END = "2026-03-31"
print("Warehouse connection ready for March 2026")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Warehouse connection ready for March 2026


## 3. Verify it with queries (grain, counts, missing values, windows)

The March 2026 verification confirms the declared daily grain: there are zero duplicate (report_date, client_hash_id, content_hash_id) groups. The slice contains 9,841,378 rows from March 1 through March 31. Of these, 413,966 rows have ga4_data_available IS TRUE; unavailable GA4 values are not treated as zero.

In [17]:
# Verification query 1: grain — duplicate daily keys should be zero.
q1 = con.sql(f"""
    SELECT COUNT(*) AS duplicate_key_groups
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_key
        FROM {TABLES['daily']}
        WHERE report_date BETWEEN DATE '{MONTH_START}' AND DATE '{MONTH_END}'
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").df()
print("1) Grain check: duplicate (date, client, content) groups")
display(q1)

# Verification query 2: March slice row count and date span.
q2 = con.sql(f"""
    SELECT COUNT(*) AS march_rows,
           MIN(report_date) AS first_report_date,
           MAX(report_date) AS last_report_date
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{MONTH_START}' AND DATE '{MONTH_END}'
""").df()
print("2) March 2026 slice count and date span")
display(q2)

# Verification query 3: availability — IS TRUE is intentional.
q3 = con.sql(f"""
    SELECT COUNT(*) AS rows_with_ga4_available_true
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{MONTH_START}' AND DATE '{MONTH_END}'
      AND ga4_data_available IS TRUE
""").df()
print("3) March 2026 rows surviving ga4_data_available IS TRUE")
display(q3)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1) Grain check: duplicate (date, client, content) groups


,duplicate_key_groups
0,0


2) March 2026 slice count and date span


,march_rows,first_report_date,last_report_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

3) March 2026 rows surviving ga4_data_available IS TRUE


,rows_with_ga4_available_true
0,413966


## 4. Data limits

Limitation: This is an unbalanced panel: clients have different tracking start dates, and GA4 coverage is much thinner than daily search coverage. The March slice contains 9,841,378 daily rows, while only 413,966 rows have ga4_data_available IS TRUE. Therefore, the GA4 feature is missing for many observations and must not be interpreted as zero. The April label also includes only pages with observed April follow-up, so the outcome frame does not represent every March page. Results are decision-support signals for refresh review, not causal claims about why traffic changed.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Feature-frame**
I use five features, all measured in the March 2026 feature window and available when the page is ranked for review: march_impressions, march_clicks, march_avg_position, content_age_days, and march_ga4_sessions. march_impressions is knowable from Search Console by the decision moment because it is accumulated during March. march_clicks is knowable for the same reason. march_avg_position is knowable from the March Search Console observations. content_age_days is knowable from the content creation date and the decision date. march_ga4_sessions is knowable only for rows where ga4_data_available IS TRUE; otherwise it is missing rather than treated as zero. The label is a separate future proxy: more than a 20% decline in April impressions compared with March impressions.

In [19]:
# Build one row per client-content page for March, then attach an April outcome label.
feature_label = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position,
        SUM(CASE WHEN ga4_data_available IS TRUE
                 THEN COALESCE(ga4_sessions, 0) ELSE NULL END) AS march_ga4_sessions,
        COUNT(*) AS march_daily_rows
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{MONTH_START}' AND DATE '{MONTH_END}'
    GROUP BY 1, 2
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS april_impressions,
        COUNT(*) AS april_daily_rows
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_avg_position,
    DATE_DIFF(
        'day',
        CAST(c.content_created_date AS DATE),
        DATE '2026-03-31'
    ) AS content_age_days,
    m.march_ga4_sessions,
    a.april_impressions,
    CAST(a.april_impressions < 0.8 * m.march_impressions AS INTEGER) AS decline_label
FROM march m
LEFT JOIN april a USING (client_hash_id, content_hash_id)
LEFT JOIN {TABLES['content']} c USING (content_hash_id)
WHERE m.march_impressions >= 100
  AND a.april_daily_rows > 0
""").df()

feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "content_age_days",
    "march_ga4_sessions",
]

model_frame = feature_label.dropna(
    subset=feature_cols + ["decline_label"]
).copy()

print(f"Feature frame rows after filters: {len(model_frame):,}")
print(f"Observed decline-label rate: {model_frame['decline_label'].mean():.3f}")
display(model_frame[feature_cols + ["decline_label"]].head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame rows after filters: 55,921
Observed decline-label rate: 0.474


,march_impressions,march_clicks,march_avg_position,content_age_days,march_ga4_sessions,decline_label
0,122.0,2.0,5.882498,42,2.0,0
1,148.0,1.0,12.350803,20,2.0,1
2,2585.0,4.0,6.251382,55,7.0,0
3,1143.0,4.0,6.104022,55,3.0,0
5,539.0,0.0,9.918178,25,2.0,1


Leakage trap: I intentionally copied the future-derived decline_label into one temporary feature named leaky_decline_label. This is not a valid feature because it is only knowable after the April outcome window. I compare a quick honest score with the deliberately leaky score, then delete the temporary column and retain the honest score.

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = model_frame[feature_cols]
y = model_frame["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        max_depth=8,
    ),
)
honest_model.fit(X_train, y_train)
honest_accuracy = accuracy_score(y_test, honest_model.predict(X_test))

# Deliberate leakage experiment: this column is copied from the future label.
leaky_frame = model_frame[feature_cols + ["decline_label"]].copy()
leaky_frame["leaky_decline_label"] = leaky_frame["decline_label"]

X_leak = leaky_frame[feature_cols + ["leaky_decline_label"]]
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leak, y, test_size=0.25, random_state=42, stratify=y
)

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        max_depth=8,
    ),
)
leaky_model.fit(Xl_train, yl_train)
leaky_accuracy = accuracy_score(yl_test, leaky_model.predict(Xl_test))

print(f"Honest accuracy: {honest_accuracy:.3f}")
print(f"Leaky accuracy:  {leaky_accuracy:.3f}")
print(f"Leakage lift:    {leaky_accuracy - honest_accuracy:+.3f}")

# Remove the invalid label-derived column from the retained frame.
model_frame = model_frame.drop(columns=["decline_label"])
assert "leaky_decline_label" not in model_frame.columns
print("Leaky feature removed. Retained features:", feature_cols)


Honest accuracy: 0.661
Leaky accuracy:  1.000
Leakage lift:    +0.339
Leaky feature removed. Retained features: ['march_impressions', 'march_clicks', 'march_avg_position', 'content_age_days', 'march_ga4_sessions']


The deliberately leaky feature produced an artificially strong score because it directly copied the future-derived label. This confirms the leakage trap: a feature that is unavailable at the decision moment can make evaluation look nearly perfect without improving real-world ranking. I removed leaky_decline_label and retained only the five pre-decision features: march_impressions, march_clicks, march_avg_position, content_age_days, and march_ga4_sessions.
4. Update the Self-check

- [x] Every section is filled with markdown reasoning and supporting code.
- [x] Exactly three verification queries are visible with outputs.
- [x] The availability check uses `ga4_data_available IS TRUE`.
- [x] The feature frame has exactly five features.
- [x] The label-derived leakage feature was deliberately tested and removed.
- [x] One limitation of the slice is named.
- [x] No client names, URLs, raw queries, or tokens appear in the notebook.
- [x] The notebook runs top to bottom without errors.
